# Descritores

Descritor é qualquer objeto que define pelo menos um destes métodos especiais:

```python
__get__(self, instance, owner)       # leitura do atributo
__set__(self, instance, value)       # escrita do atributo
__delete__(self, instance)           # deleção do atributo
# opcional, chamado ao criar a classe
__set_name__(self, owner, name)      

```

| TYPE                | PROTOCOL METHODS       | USE CASE                                  |
|---------------------|------------------------|-------------------------------------------|
| property            | `__get__`, `__set__`, `__delete__` | Managed attributes with getters/setters  |
| Classmethod         | `__get__`                | Class-bound methods                      |
| staticmethod        | `__get__`                | Utility methods (no class/instance binding) |
| Custom Data Desc.   | `__get__`, `__set__`       | Reusable attribute logic (e.g., validation) |
| Custom Non-Data Desc. | `__get__`              | Read-only/computed attributes            |
| `__slots__`           | Implicit descriptors   | Memory optimization, restricted attributes |

Eles permitem controlar o acesso a atributos em classes.

Dois tipos:

- **Data descriptor**: tem __set__ e/ou __delete__. Tem prioridade sobre instance.__dict__.
- **Non-data descriptor**: só tem __get__. Pode ser “sobreposto” por uma chave homônima no __dict__ da instância.

Regras de resolução (resumo):

1. data descriptor > obj.__dict ↓ · non-data descriptor > atributo de classe.

# Função definida na classe (método de instância) – non-data descriptor

Funções viram métodos ligados graças a `__get__`.

In [1]:
class A:
    def f(self):
        return f"Olá de {self!r}"

a = A()
print(type(A.__dict__['f']))   # <class 'function'>
print(a.f())                   # método ligado: imprime "Olá de <A ...>"
print(A.f(a))                  # chamada manual passando a instância


<class 'function'>
Olá de <__main__.A object at 0x0000022B8EDB9780>
Olá de <__main__.A object at 0x0000022B8EDB9780>


# `property` – data descriptor (getter/setter/deleter)

Controla acesso e prevalece sobre `__dict__` da instância.

In [2]:
class Celsius:
    def __init__(self, t):
        self._t = float(t)

    @property
    def t(self):
        return self._t

    @t.setter
    def t(self, v):
        self._t = float(v)

c = Celsius(10)
c.t = 21
print(c.t)                     # 21.0

# Mesmo “injetando” uma chave no __dict__, a property vence (data descriptor)
c.__dict__['t'] = 'hack'
print(c.t)                     # 21.0


21.0
21.0


# `classmethod` – non-data descriptor (liga ao cls)

Retorna uma função ligada à classe, não à instância.

In [3]:
class C:
    count = 0

    @classmethod
    def inc(cls):
        cls.count += 1

C.inc()
print(C.count)                 # 1


1


# `staticmethod` – non-data descriptor (sem binding)

Não liga nem a `self` nem a `cls`.

In [4]:
class Math:
    @staticmethod
    def add(a, b):
        return a + b

print(Math.add(2, 3))          # 5
print(Math().add(2, 3))        # 5


5
5


# `functools.cached_property` – non-data descriptor (cacheia no `__dict__`)

Calcula uma vez, guarda no `__dict__` da instância.

In [5]:
from functools import cached_property

class Expensive:
    calls = 0

    @cached_property
    def value(self):
        Expensive.calls += 1
        return 42

e = Expensive()
print(e.value, e.value)        # 42 42 (só computa uma vez)
print(Expensive.calls)         # 1
print(e.__dict__['value'])     # 42 (foi salvo no __dict__)


42 42
1
42


# `__slots__` → member_descriptor – data descriptor

Atributos de `__slots__` são descritores que guardam valores fora do `__dict__`.

In [6]:
class P:
    __slots__ = ('x',)
    def __init__(self, x):
        self.x = x

p = P(3)
print(type(P.__dict__['x']))   # <class 'member_descriptor'>
try:
    p.y = 10                   # não permite novos atributos
except AttributeError as e:
    print(e)


<class 'member_descriptor'>
'P' object has no attribute 'y'


# Métodos embutidos (`method_descriptor`) – non-data descriptor

Métodos C de tipos built-in (ex.: `list.append`).

In [7]:
print(type(list.append))       # <class 'method_descriptor'>
lst = [1]
list.append(lst, 2)            # chamada como função C passando a lista
print(lst)                     # [1, 2]


<class 'method_descriptor'>
[1, 2]


# Operadores especiais (`wrapper_descriptor`)

Implementações C de dunders (ex.: `int.__add__`).

In [8]:
print(type(int.__add__))       # <class 'wrapper_descriptor'>
print(int.__add__(10, 5))      # 15
print((10).__add__(5))         # 15


<class 'wrapper_descriptor'>
15
15


# Atributos C expostos com `getset_descriptor`

Muitos atributos “só de leitura” de tipos C são get/set em C (ex.: `datetime.datetime.year`).
Outro exemplo clássico: o próprio `__dict__` de classes é exposto por um getset do metaclasse `type`, e retorna um `mappingproxy`.

In [10]:
import datetime
print(type(datetime.datetime.year))             # <class 'getset_descriptor'>
dt = datetime.datetime(2025, 8, 25)
print(datetime.datetime.year.__get__(dt))       # 2025 (acesso via __get__)

<class 'getset_descriptor'>
2025


In [11]:
# O dicionário de uma classe é um mappingproxy, exposto por um getset do 'type'
class X: 
    a = 1
print(type(type.__dict__['__dict__']))          # <class 'getset_descriptor'>
print(type(X.__dict__))                         # <class 'mappingproxy'>

<class 'getset_descriptor'>
<class 'mappingproxy'>


# Criando seu próprio descritor (validador)

Uma forma de DRY de nao repetir codigo de validacao usando o descritor `property`

Exemplo de data descriptor que aceita apenas `int` e usa `__set_name__` para saber o nome do atributo:

In [12]:
class IntegerField:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:     # acesso pela classe
            return self
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        if not isinstance(value, int):
            raise TypeError(f"{self.name} deve ser int")
        instance.__dict__[self.name] = value

class Point:
    x = IntegerField()
    y = IntegerField()
    def __init__(self, x, y):
        self.x = x
        self.y = y

p = Point(10, 20)
print(p.x, p.y)                 # 10 20
# p.x = 'ops'                   # TypeError: x deve ser int


10 20


In [14]:
try:
    p.x = 'ops'
except TypeError as type_e:
    print(f"falha TypeError: {type_e}")

falha TypeError: x deve ser int


## DRY mode: usando funcao fabricante 

In [ ]:
def int_property(private_name):
    def getter(self):
        return getattr(self, private_name, None)
    def setter(self, value):
        if not isinstance(value, int):
            # derive public name from private (e.g. '_x' -> 'x')
            name = private_name[1:] if private_name.startswith('_') else private_name
            raise TypeError(f"{name} must be an integer.")
        setattr(self, private_name, value)
    def deleter(self):
        setattr(self, private_name, None)
    return property(getter, setter, deleter)

class Point:
    x = int_property("_x")
    y = int_property("_y")

    def __init__(self, x, y):
        self._x = None
        self._y = None
        self.x = x
        self.y = y


## usando implementacao pelo `@property`

In [ ]:
class Point:
    def __init__(self, x, y):
        self._x = None
        self._y = None
        self.x = x
        self.y = y

    @property
    def x(self) -> int:
        return self._x

    @x.setter
    def x(self, value: int) -> None:
        if not isinstance(value, int):
            raise TypeError("x must be an integer.")
        self._x = value

    @x.deleter
    def x(self) -> None:
        self._x = None

    @property
    def y(self) -> int:
        return self._y

    @y.setter
    def y(self, value: int) -> None:
        if not isinstance(value, int):
            raise TypeError("y must be an integer.")
        self._y = value

    @y.deleter
    def y(self) -> None:
        self._y = None


# Descritores OS-related 

| Resource              | How to get it in Python                                               | Notes                                                                                             |                                                       |
| --------------------- | --------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------- | ----------------------------------------------------- |
| Regular file          | `os.open()`, or `open(...).fileno()`                                  | `os.open` gives a raw FD (int). `open()` gives a Python file object; `f.fileno()` returns its FD. |                                                       |
| Socket                | `socket.socket()`, `.fileno()`                                        | Also `socket.socketpair()` (Unix + Windows ≥ 3.8).                                                |                                                       |
| Pipe (anonymous)      | `os.pipe()` → `(rfd, wfd)`                                            | Unidirectional; great for parent/child or threads.                                                |                                                       |
| FIFO / named pipe     | `os.mkfifo()` (Unix), Windows named pipes via `pywin32`               | FIFOs persist in the filesystem (Unix).                                                           |                                                       |
| TTY / PTY             | `os.open('/dev/tty', ...)`, `pty.openpty()` (Unix)                    | Terminals and pseudo-terminals.                                                                   |                                                       |
| Devices               | `os.open('/dev/null', ...)`, `os.open('/dev/random', ...)`            | Special files (Unix).                                                                             |                                                       |
| Directory FD          | \`os.open('dir', os.O\_RDONLY                                         | os.O\_DIRECTORY)\` (Unix)                                                                         | Useful for `openat`-style ops via `os.openat` (Unix). |
| Event sources (Linux) | `inotify_init`, `eventfd`, `timerfd`, `signalfd` via `ctypes` or libs | Advanced/Unix-specific.                                                                           |                                                       |
| Windows HANDLEs       | `msvcrt.get_osfhandle(fd)`, `msvcrt.open_osfhandle(h, flags)`         | Convert between CRT fd and OS HANDLE.                                                             |                                                       |
